# NB3 · Building and measuring a model

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---


## What this notebook does

Building the model is the shortest task in this notebook; it takes a few lines. The rest
is given over to measuring how much that model is worth.

Measuring takes longer for this reason: it is easy for a model to look good and hard to
tell whether it is good. You will see both here.


## Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'en'

print('Ready.')


---

## Code carried from the previous notebook

Paste the whole block collected at the end of the previous notebook into the cell
below. Do not delete the `#@cdss` marker on the first line; that block is collected
again at the end of this notebook and carried to the next one.

Running the block rebuilds everything you wrote in the earlier notebooks. Where it
reads data from the web, the cell may take a few seconds.


In [ ]:
#@cdss onceki_defter
# Paste the generated code below this line.


### Check · The carried code


In [ ]:
kit.check_defined('X_train', 'X_test', 'y_train', 'y_test',
                  'train', 'test', 'RANDOM_SEED')


---

## Step 1 · Building and teaching the model

A model is a calculation method that extracts patterns from data. We begin with a simple
one.

The reason for starting simple is not performance but three properties. What a simple
model looks at can be read; it produces probabilities; and it is less inclined to
memorise on little data. A more complex model may beat it, and the yardstick stays in
our hands.

Note one point. Where the target condition is a minority in the cohort, a model may take
the easy route and say negative to everyone. The prompt asks for balanced class weights to
prevent this.


### Prompt 1

```
Build and teach a model using X_train and y_train.

Use a simple and interpretable classification method. Where the target condition is a
minority, stop the model drifting to the majority; weight the classes in a balanced way.

Use RANDOM_SEED for reproducibility. Do not search for settings yet; this first model is
built to be beaten.

Keep the model under the name model. Print which method you chose and why it is a
reasonable starting point for this problem, in two sentences.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a trained model named model.
The model must be able to give the probability of a patient reaching the target
condition: a number between 0 and 1, not only a 0 or 1 label.
```


In [ ]:
#@cdss model
# Paste the generated code below this line.


### Check 1


In [ ]:
kit.check_model('model', sample=X_test)


### Python note · Objects and methods

In the code you will see a line such as `model = LogisticRegression(...)` followed by
`model.fit(...)`. Two ideas appear here.

The first line creates an **object**. An object holds both information and the operations
that can be performed with it. Think of a patient file: it contains data, and certain
operations on that file are defined.

`model.fit(...)` calls a **method** of the object. The name after the dot is something
that object can do. `fit` learns, `predict` predicts and `predict_proba` produces
probabilities.

This structure is the same across machine learning libraries. Change the method and the
names `fit` and `predict` stay the same, which is why swapping one model for another is
straightforward.


---

## Step 2 · Producing predictions

The model can now produce predictions for the patients in the test group.

The prediction has to be a probability rather than a class label. A model that returns
class labels has fixed the threshold at its own default, usually fifty percent. That
default is not a clinical decision.

You choose the threshold, and the choice rests on clinical reasoning. Where a miss is
costly the threshold is lowered and more alerts are produced. Where a false alarm is
costly the opposite is done. You will see this in step three.


### Prompt 2

```
Produce predictions for the patients in the test group with the trained model.

Ask for probabilities rather than class labels: for each patient, the probability of
reaching the target condition, as a number between 0 and 1.

Keep the result under the name probability. Print the lowest, highest and mean
probability.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a sequence of numbers named probability.
Its length must equal the number of patients in the test group.
The values must lie between 0 and 1 and must not all be identical.
```


In [ ]:
#@cdss tahmin
# Paste the generated code below this line.


### Check 2


In [ ]:
kit.check_numbers(probability, name='probability', count=len(y_test), low=0.0, high=1.0)


---

## Step 3 · Honest measurement

Now the substantive work. The performance of a model cannot be conveyed by a single
number; six headings are read together.

**Accuracy misleads.** If the target condition occurs in ten percent of patients, a rule
that says negative to everyone is ninety percent accurate. Accuracy there measures how
rare the condition is, not what the model does.

**Discrimination alone is not enough.** Ranking patients correctly and returning a
probability that corresponds to reality are different things. If you are going to set a
threshold, the second matters more.

**The real question is the clinical translation.** How many alerts fire per hundred
patients and how many are true? That line is the counterpart, in your own model, of the
Epic Sepsis Model example from the lecture on 16 September.


### Prompt 3

```
Write a piece of work that measures the performance of the model. Name it
measure_performance and let it take the true outcomes, the predicted probabilities and a
threshold.

Have it compute the following, writing under each what it means in plain language:
1. How well the model ranks patients. Give this with a confidence interval; a single
   number is not enough.
2. How well the probabilities the model returns correspond to reality.
3. At the given threshold: the rate at which affected patients are caught, the rate at
   which unaffected patients avoid an alert, the rate at which an alert turns out to be
   justified, and the rate at which the absence of an alert turns out to be justified.
4. Clinical translation: at this threshold, how many alerts fire per hundred patients,
   how many are true, and how many cases are missed entirely.
5. The same measurements broken down by sex group. Where a group has very few patients,
   produce no figure; write insufficient sample instead.
6. What a rule that learns nothing, giving everyone the same answer, would achieve on
   the same measures.

Print the results and also give them back as a dictionary.

Then run this piece of work on the test group with a threshold of 0.50 and keep the
result under the name result.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a runnable piece of work named measure_performance.
There must be a dictionary named result containing the keys:
  discrimination, calibration, sensitivity, specificity, ppv, npv,
  alerts_per_hundred, true_alerts_per_hundred, null_rule_accuracy
```


In [ ]:
#@cdss olcum
# Paste the generated code below this line.


### Check 3


In [ ]:
kit.check_function('measure_performance')

required = ['discrimination', 'calibration', 'sensitivity', 'specificity', 'ppv', 'npv',
            'alerts_per_hundred', 'true_alerts_per_hundred', 'null_rule_accuracy']
missing = [k for k in required if k not in result]
print('Missing keys in result:', missing if missing else 'none')


### The accuracy trap

The two cells below are supplied. Run the first and read the figure, then run the
second.


In [ ]:
import numpy as _np
accuracy = float(((_np.asarray(probability) >= 0.5).astype(int) == _np.asarray(y_test)).mean())
print(f'Accuracy of the model: {accuracy:.1%}')


In [ ]:
rate = float(_np.asarray(y_test).mean())
null = max(rate, 1 - rate)
print(f'Accuracy of a rule that answers the same for everyone: {null:.1%}')
print()
print('The gap between the two is what the model actually contributes.')


### Python note · Dictionaries

The prompt asked for the results to be given back as a **dictionary**. A dictionary stores
each value under a name: writing `result['sensitivity']` reaches that value.

Unlike a list, it is accessed by name rather than by position. Had nine values been
returned as a list, you would have had to remember what the third one was. In a dictionary
the name is written down.

The choice matters in clinical code: in NB5 you will pull values from this dictionary
while writing the compliance report, and clear names will serve you there.


### Reading the result

Look at four things.

**The confidence interval.** If it includes 0.5, the model cannot be distinguished from
chance, whatever the number in the middle says.

**The null comparison.** If the accuracy of the model and of the rule that answers the
same for everyone are close, the model has learned little worth having.

**The clinical translation.** Look at how many alerts fire per hundred patients and how
many are true. Where most alerts are false, the system is switched off in the clinic
before long.

**The subgroups.** Where a group reads insufficient sample, that is a finding rather than
a shortcoming: a system cannot be shown to be fair for a group it was never tested on.

On a cohort of one hundred patients an unfavourable result is expected. Reaching that
judgement about a system you built yourself is a different experience from hearing about
someone else's failure.


---

## End of notebook · Collecting the code

The cell below collects the code you carried from the earlier notebooks together with
what you added here, as one block. Copy the whole block; you will paste it into the
first cell of NB4.

The block is also saved as `cdss_nb3.py`. That file disappears when the Colab session closes,
so keep a copy in a text file on your own computer as well.


In [ ]:
code_so_far = kit.export(save_as='cdss_nb3.py')


## What this notebook did

Model, prediction and measurement layers were added to the system. The measurement
produces a dictionary of nine values rather than a single number, and includes the
clinical translation.

---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The dataset
you used is an open collection prepared for teaching and does not represent the patient
population of your own institution. The material is for teaching.
